## 第1步：读 CSV 文件

In [9]:
import csv

# 读取 CSV 文件 bom_list.csv
with open("bom_list.csv", mode="r",encoding="utf-8") as file:
    reader = csv.DictReader(file) # DictReader 会把每一行转成字典,这里reader是一个迭代器(DictReader的实例)而非字典
    boms = list(reader) # 转化为列表

for bom in boms:
    print(bom)
    

[{'bom_id': 'BOM-001', 'material': '螺丝 M3', 'status': '待审核', 'owner': '张三'}, {'bom_id': 'BOM-002', 'material': '钣金件 A12', 'status': '已通过', 'owner': '李四'}, {'bom_id': 'BOM-003', 'material': '电池 5000mAh', 'status': '待审核', 'owner': '王五'}, {'bom_id': 'BOM-004', 'material': '外壳 B 款', 'status': '已驳回', 'owner': '赵六'}, {'bom_id': 'BOM-005', 'material': 'PCB 主板', 'status': '待审核', 'owner': '钱七'}]
{'bom_id': 'BOM-001', 'material': '螺丝 M3', 'status': '待审核', 'owner': '张三'}
{'bom_id': 'BOM-002', 'material': '钣金件 A12', 'status': '已通过', 'owner': '李四'}
{'bom_id': 'BOM-003', 'material': '电池 5000mAh', 'status': '待审核', 'owner': '王五'}
{'bom_id': 'BOM-004', 'material': '外壳 B 款', 'status': '已驳回', 'owner': '赵六'}
{'bom_id': 'BOM-005', 'material': 'PCB 主板', 'status': '待审核', 'owner': '钱七'}


## 第2步：筛选状态=“待审核”的行

In [29]:
pending_boms = []
for bom in boms:
    if bom["status"] == "待审核":
        pending_boms.append(bom)

print("=== 待审核的 BOM ===")
for bom in pending_boms:
    print(f'{bom["bom_id"]} | {bom["material"]} | {"bom[owner]"}') # f代表格式化，f""中{}代表占位符，注意"和'区分用：f""则bom['bom_id']

=== 待审核的 BOM ===
BOM-001 | 螺丝 M3 | bom[owner]
BOM-003 | 电池 5000mAh | bom[owner]
BOM-005 | PCB 主板 | bom[owner]


## 第3步：调天气 API（巩固 requests）

In [43]:
import requests

# wttr.in 是一个无需 Key 的天气 API
city = "南京"
url = f"https://wttr.in/{city}?format=%C+%t"

response = requests.get(url)
print(f"{city}的天气：{response.text}")

南京的天气：Sunny +26°C


## 第4步：批量工单分类 + 导出 CSV
这是第一周最有价值的综合练习。
把多条工单批量发给 DeepSeek，结果保存到新 CSV。

In [70]:
import requests
import time
import csv

# --   配置   --
api_key = "[REDACTED]"
url = "https://openrouter.ai/api/v1/chat/completions"
headers = {
    "Authorization": f"Bearer {api_key}",
    "Content-Type": "application/json",
}
# --  调用ai 给工单分类
def classify_ticket(description):
    data = json.dumps({
        "model": "nvidia/nemotron-3.5-lightning:free",
        "messages":[
            {"role":"system", "content":"你是一个工单分类助手，只返回分类名称：权限/流程/接口/数据"},
            {"role":"user", "content": description}
        ],
        "reasoning": {"enabled": True}
    })
    try:
        resp = requests.post(url=url, headers=headers, data=data)
        resp = resp.json()["choices"][0]["message"]["content"]
#         print(resp)
        return resp
    except Exception as e:
        return f"错误：{e}"

# 准备待分类的工单列表
ticket_list = [
    {"id": "TK-001", "description": "我提交了报销单，财务说看不到"},
    {"id": "TK-002", "description": "ERP 系统打不开"},
    {"id": "TK-003", "description": "申请开通 SAP 权限"},
    {"id": "TK-004", "description": "加班审批流卡住了"},
    {"id": "TK-005", "description": "BOM 变更后物料清单没更新"}
]

# category = classify_ticket("我提交了报销单，财务说看不到")

# 批量分类
result_list = []
for ticket in ticket_list:
    category = classify_ticket(ticket["description"])
    result_list.append({
        "ticket_id": ticket["id"],
        "ticket_desc": ticket["description"],
        "category": category
    })
    time.sleep(0.5)# 暂停0.5秒,防止调用太频繁被限流

for result in result_list:
    print(result)
    
# 保存到 CSV
with open("classified_tickets.csv", mode="w",newline="",encoding="utf-8-sig") as file:
    fieldnames = ["ticket_id", "ticket_desc", "category"]
    writer = csv.DictWriter(file, fieldnames = fieldnames)
    writer.writeheader()
    writer.writerows(result_list)

print("\n分类完成！结果已保存到 classified_tickets.csv")

{'ticket_id': 'TK-001', 'ticket_desc': '我提交了报销单，财务说看不到', 'category': '流程'}
{'ticket_id': 'TK-002', 'ticket_desc': 'ERP 系统打不开', 'category': 'permission'}
{'ticket_id': 'TK-003', 'ticket_desc': '申请开通 SAP 权限', 'category': '权限'}
{'ticket_id': 'TK-004', 'ticket_desc': '加班审批流卡住了', 'category': '流程'}
{'ticket_id': 'TK-005', 'ticket_desc': 'BOM 变更后物料清单没更新', 'category': '数据'}

分类完成！结果已保存到 classified_tickets.csv
